In [1]:
# Linear Regression exercises using Advertising dataset from ISLP
# Goal: predict sales from tv, radio, newspaper advertising
# File created: 25 November 2026
# Last edited: 25 November 2026

In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from scipy.stats import norm

np.set_printoptions(suppress = True)
import warnings
warnings.filterwarnings("ignore")

In [3]:
data = pd.read_csv("Datasets/Advertising.csv")
data.head()

,Unnamed: 0,TV,radio,newspaper,sales
0,1,230.1,37.8,69.2,22.1
1,2,44.5,39.3,45.1,10.4
2,3,17.2,45.9,69.3,9.3
3,4,151.5,41.3,58.5,18.5
4,5,180.8,10.8,58.4,12.9


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  200 non-null    int64  
 1   TV          200 non-null    float64
 2   radio       200 non-null    float64
 3   newspaper   200 non-null    float64
 4   sales       200 non-null    float64
dtypes: float64(4), int64(1)
memory usage: 7.9 KB


In [5]:
data.columns

Index(['Unnamed: 0', 'TV', 'radio', 'newspaper', 'sales'], dtype='object')

In [6]:
# Predicting "sales" from "TV" using sklearn.linear_model's LinearRegression - see ISLP pages 71-77
print("Simple Linear Regression: sales = b0 + b1 x TV + eps")
Y = data["sales"].to_numpy()
X = data["TV"].to_numpy().reshape((-1, 1))
print(X.shape, Y.shape)
model = LinearRegression()
model.fit(X, Y)
beta0 = model.intercept_
beta1 = model.coef_.item()
print("beta0 = %s, beta1 = %s" % (beta0, beta1))
Rsquare = model.score(X, Y)
print("Rsquare = %s" % Rsquare)

Simple Linear Regression: sales = b0 + b1 x TV + eps
(200, 1) (200,)
beta0 = 7.0325935491276885, beta1 = 0.04753664043301979
Rsquare = 0.611875050850071


In [7]:
# Predicting "sales" from "radio" - see ISLP page 80
print("Simple Linear Regression: sales = b0 + b1 x radio + eps")
Y = data["sales"].to_numpy()
X = data["radio"].to_numpy().reshape((-1, 1))
print(X.shape, Y.shape)
model = LinearRegression()
model.fit(X, Y)
beta0 = model.intercept_
beta1 = model.coef_.item()
print("beta0 = %s, beta1 = %s" % (beta0, beta1))
Rsquare = model.score(X, Y)
print("Rsquare = %s" % Rsquare)

Simple Linear Regression: sales = b0 + b1 x radio + eps
(200, 1) (200,)
beta0 = 9.311638095158285, beta1 = 0.20249578339243957
Rsquare = 0.33203245544529536


In [8]:
# Predicting "sales" from "newspaper" - see ISLP page 80
print("Simple Linear Regression: sales = b0 + b1 x newspaper + eps")
Y = data["sales"].to_numpy()
X = data["newspaper"].to_numpy().reshape((-1, 1))
print(X.shape, Y.shape)
model = LinearRegression()
model.fit(X, Y)
beta0 = model.intercept_
beta1 = model.coef_.item()
print("beta0 = %s, beta1 = %s" % (beta0, beta1))
Rsquare = model.score(X, Y)
print("Rsquare = %s" % Rsquare)

Simple Linear Regression: sales = b0 + b1 x newspaper + eps
(200, 1) (200,)
beta0 = 12.35140706927816, beta1 = 0.05469309847227336
Rsquare = 0.05212044544430516


In [15]:
class Jie_SimpleLinearRegression:
    
    def __init__(self):
        self.beta0 = None
        self.beta1 = None
        self.sigma2 = None
        self.Rsquare = None
        self.beta0_stdev = None
        self.beta1_stdev = None
        self.beta0_t = None
        self.beta1_t = None
        self.beta0_p = None
        self.beta1_p = None
        self.results = None
    
    def fit(self, X, Y, predictor_name):
        
        Xc = np.insert(X, 0, np.ones(len(X)), axis = 1)
        self.beta0, self.beta1 = np.linalg.inv(Xc.T @ Xc) @ Xc.T @ Y
        
        Yhat = self.predict(X)
        self.sigma2 = (np.linalg.norm(Y - Yhat) ** 2) / (len(Y) - 2)
        
        beta_var = np.diag(self.sigma2 * np.linalg.inv(Xc.T @ Xc))
        self.beta0_stdev, self.beta1_stdev = np.sqrt(beta_var)

        self.beta0_t = self.beta0 / self.beta0_stdev
        self.beta1_t = self.beta1 / self.beta1_stdev
        
        self.beta0_p = 2 * norm.sf(np.abs(self.beta0_t))
        self.beta1_p = 2 * norm.sf(np.abs(self.beta1_t))

        beta = np.array([self.beta0, self.beta1])
        beta_stdev = np.array([self.beta0_stdev, self.beta1_stdev])
        beta_t = np.array([self.beta0_t, self.beta1_t])
        beta_p = np.array([self.beta0_p, self.beta1_p])
        self.results = pd.DataFrame(zip(beta, beta_stdev, beta_t, beta_p),
                                    columns = ["beta", "beta stdev", "beta t", "beta p"],
                                    index = ["intercept", predictor_name])
        print(self.results)
        
    def score(self, X, Y):
        if (self.beta0 is None or self.beta1 is None):
            print("Model untrained/not fitted!")
            return None
        Yhat = self.predict(X)
        RSS = np.linalg.norm(Y - Yhat) ** 2
        TSS = np.linalg.norm(Y - np.mean(Y)) ** 2
        self.Rsquare = 1 - RSS/TSS
        print("Rsquare = %s" % self.Rsquare)
        
    def predict(self, X):
        if (self.beta0 is None or self.beta1 is None):
            print("Model untrained/not fitted!")
            return None
        Yhat = (self.beta0 + self.beta1 * X).reshape(-1)
        return Yhat

In [16]:
# Predicting "sales" from "TV" by hand - see ISLP pages 71-77
print("Simple Linear Regression: sales = b0 + b1 x TV + eps")
Y = data["sales"].to_numpy()
predictor_name = "TV"
X = data[predictor_name].to_numpy().reshape((-1, 1))
print(X.shape, Y.shape)
model = Jie_SimpleLinearRegression()
model.fit(X, Y, predictor_name)
model.score(X, Y)

Simple Linear Regression: sales = b0 + b1 x TV + eps
(200, 1) (200,)
               beta  beta stdev     beta t        beta p
intercept  7.032594    0.457843  15.360275  3.022888e-53
TV         0.047537    0.002691  17.667626  7.446586e-70
Rsquare = 0.6118750508500712
